In [82]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================
# LOAD SAVED DEMAND LSTM
# ============================================

MODEL_FILE = "Demand_LSTM.keras"
X_SCALER_FILE = "Demand_LSTM_X_Scaler.pkl"
Y_SCALER_FILE = "Demand_LSTM_y_Scaler.pkl"
HP_FILE = "Demand_LSTM_Hyperparameters.json"

model = load_model(MODEL_FILE)

X_scaler = joblib.load(X_SCALER_FILE)
y_scaler = joblib.load(Y_SCALER_FILE)

with open(HP_FILE, "r") as f:
    hp = json.load(f)

LOOKBACK = int(hp["lookback"])

print("========================================")
print("MODEL LOADED")
print("========================================")
print("Lookback      :", LOOKBACK)
print("Units         :", hp["units"])
print("Layers        :", hp["layers"])
print("Dropout       :", hp["dropout"])
print("Dense Units   :", hp["dense_units"])
print("Learning Rate :", hp["learning_rate"])
print("Batch Size    :", hp["batch_size"])
print("Epochs        :", hp["epochs"])
print("Seed          :", hp["seed"])

MODEL LOADED
Lookback      : 3
Units         : 64
Layers        : 2
Dropout       : 0.0
Dense Units   : 8
Learning Rate : 0.001
Batch Size    : 16
Epochs        : 40
Seed          : 42


In [83]:
# ============================================
# LOAD DATA
# ============================================

df = pd.read_excel("Book1.xlsx")

df["Date"] = pd.to_datetime(df["Date"])

df = (
    df.sort_values("Date")
      .reset_index(drop=True)
)

print("Data shape:", df.shape)
print("Start:", df["Date"].min())
print("End  :", df["Date"].max())

print("\nColumns:")
print(df.columns.tolist())

Data shape: (15, 7)
Start: 2025-01-01 00:00:00
End  : 2026-03-01 00:00:00

Columns:
['Date', 'Electricity_Requirement', 'Temperature', 'Rainfall', 'Humidity', 'Festival', 'Solar_Irradiance']


In [84]:
df.head()

,Date,Electricity_Requirement,Temperature,Rainfall,Humidity,Festival,Solar_Irradiance
0,2025-01-01,9795,23.86,31.285429,80.46,1,268.909644
1,2025-02-01,10242,25.73,10.476571,68.88,0,252.323688
2,2025-03-01,11864,28.23,59.660286,66.46,0,289.310912
3,2025-04-01,11892,28.97,97.794000,71.18,0,303.305385
4,2025-05-01,11332,28.83,146.711714,73.94,0,224.889697


In [85]:
# ============================================
# FEATURE ENGINEERING
# ============================================

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

# Cyclical month features
df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

# Historical demand features
df["Demand_Lag_1"] = (
    df["Electricity_Requirement"].shift(1)
)

df["Demand_Lag_2"] = (
    df["Electricity_Requirement"].shift(2)
)

df["Demand_Lag_3"] = (
    df["Electricity_Requirement"].shift(3)
)

# Historical rolling features
df["Demand_Rolling_3"] = (
    df["Electricity_Requirement"]
      .shift(1)
      .rolling(3)
      .mean()
)

df["Demand_Rolling_6"] = (
    df["Electricity_Requirement"]
      .shift(1)
      .rolling(6)
      .mean()
)

df["Demand_Rolling_12"] = (
    df["Electricity_Requirement"]
      .shift(1)
      .rolling(12)
      .mean()
)

FEATURES = [
    "Humidity",
    "Rainfall",
    "Solar_Irradiance",
    "Temperature",
    "Year",
    "Month_sin",
    "Month_cos",
    "Festival",
    "Demand_Lag_1",
    "Demand_Lag_2",
    "Demand_Lag_3",
    "Demand_Rolling_3",
    "Demand_Rolling_6",
    "Demand_Rolling_12"
]

print("\nFeature count:", len(FEATURES))
print(FEATURES)


Feature count: 14
['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Festival', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3', 'Demand_Rolling_3', 'Demand_Rolling_6', 'Demand_Rolling_12']


In [86]:
df.columns

Index(['Date', 'Electricity_Requirement', 'Temperature', 'Rainfall',
       'Humidity', 'Festival', 'Solar_Irradiance', 'Year', 'Month',
       'Month_sin', 'Month_cos', 'Demand_Lag_1', 'Demand_Lag_2',
       'Demand_Lag_3', 'Demand_Rolling_3', 'Demand_Rolling_6',
       'Demand_Rolling_12'],
      dtype='object')

In [87]:
FEATURES = [
    "Humidity",
    "Rainfall",
    "Solar_Irradiance",
    "Temperature",
    "Year",
    "Month_sin",
    "Month_cos",
    "Festival",
    "Demand_Lag_1",
    "Demand_Lag_2",
    "Demand_Lag_3",
    "Demand_Rolling_3",
    "Demand_Rolling_6",
    "Demand_Rolling_12"
]

print("Feature count:", len(FEATURES))
print("Features:", FEATURES)

print("\n2026 rows:")
print(
    df[df["Date"].dt.year == 2026][
        ["Date", "Electricity_Requirement"]
    ]
)

Feature count: 14
Features: ['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Festival', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3', 'Demand_Rolling_3', 'Demand_Rolling_6', 'Demand_Rolling_12']

2026 rows:
         Date  Electricity_Requirement
12 2026-01-01                    10067
13 2026-02-01                    10125
14 2026-03-01                    12233


In [88]:
print("Date range:")
print(df["Date"].min())
print(df["Date"].max())

print("\nLast 20 rows:")
print(df[["Date", "Electricity_Requirement"]].tail(20))

print("\nDate dtype:")
print(df["Date"].dtype)

Date range:
2025-01-01 00:00:00
2026-03-01 00:00:00

Last 20 rows:
         Date  Electricity_Requirement
0  2025-01-01                     9795
1  2025-02-01                    10242
2  2025-03-01                    11864
3  2025-04-01                    11892
4  2025-05-01                    11332
5  2025-06-01                    11514
6  2025-07-01                    12382
7  2025-08-01                    11271
8  2025-09-01                    10972
9  2025-10-01                    10087
10 2025-11-01                    10076
11 2025-12-01                    10150
12 2026-01-01                    10067
13 2026-02-01                    10125
14 2026-03-01                    12233

Date dtype:
datetime64[ns]


In [89]:
# ============================================
# FIX DATE FORMAT
# ============================================

df["Date"] = pd.to_datetime(df["Date"])

# Convert every date to month-start
df["Date"] = df["Date"].dt.to_period("M").dt.to_timestamp()

df = (
    df.sort_values("Date")
      .drop_duplicates("Date")
      .reset_index(drop=True)
)

print("Start:", df["Date"].min())
print("End  :", df["Date"].max())

# ============================================
# SELECT EXACTLY JAN-MAR 2026
# ============================================

future_actual = df[
    df["Date"].isin(
        pd.to_datetime([
            "2026-01-01",
            "2026-02-01",
            "2026-03-01"
        ])
    )
][["Date", "Electricity_Requirement"]].copy()

print("\nSelected 2026 rows:")
print(future_actual)

if len(future_actual) != 3:
    raise ValueError(
        f"Expected exactly 3 unique months, found {len(future_actual)}"
    )

Start: 2025-01-01 00:00:00
End  : 2026-03-01 00:00:00

Selected 2026 rows:
         Date  Electricity_Requirement
12 2026-01-01                    10067
13 2026-02-01                    10125
14 2026-03-01                    12233


In [90]:
df.tail(3)

,Date,Electricity_Requirement,Temperature,Rainfall,Humidity,Festival,Solar_Irradiance,Year,Month,Month_sin,Month_cos,Demand_Lag_1,Demand_Lag_2,Demand_Lag_3,Demand_Rolling_3,Demand_Rolling_6,Demand_Rolling_12
12,2026-01-01,10067,25.5,38.5,73.5,1,200.0,2026,1,0.500000,8.660254e-01,10150.0,10076.0,10087.0,10104.333333,10823.000000,10964.750000
13,2026-02-01,10125,26.5,12.2,69.5,0,225.0,2026,2,0.866025,5.000000e-01,10067.0,10150.0,10076.0,10097.666667,10437.166667,10987.416667
14,2026-03-01,12233,28.5,33.7,67.5,0,241.0,2026,3,1.000000,6.123234e-17,10125.0,10067.0,10150.0,10114.000000,10246.166667,10977.666667


In [91]:
future_2026 = df[
    (df["Date"] >= "2026-01-01") &
    (df["Date"] <= "2026-03-01")
].copy()

print(future_2026)

         Date  Electricity_Requirement  Temperature  Rainfall  Humidity  \
12 2026-01-01                    10067         25.5      38.5      73.5   
13 2026-02-01                    10125         26.5      12.2      69.5   
14 2026-03-01                    12233         28.5      33.7      67.5   

    Festival  Solar_Irradiance  Year  Month  Month_sin     Month_cos  \
12         1             200.0  2026      1   0.500000  8.660254e-01   
13         0             225.0  2026      2   0.866025  5.000000e-01   
14         0             241.0  2026      3   1.000000  6.123234e-17   

    Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  Demand_Rolling_3  \
12       10150.0       10076.0       10087.0      10104.333333   
13       10067.0       10150.0       10076.0      10097.666667   
14       10125.0       10067.0       10150.0      10114.000000   

    Demand_Rolling_6  Demand_Rolling_12  
12      10823.000000       10964.750000  
13      10437.166667       10987.416667  
14      10246.16666

In [92]:
jan_2026 = df[
    df["Date"] == "2026-01-01"
].copy()

print(jan_2026)

         Date  Electricity_Requirement  Temperature  Rainfall  Humidity  \
12 2026-01-01                    10067         25.5      38.5      73.5   

    Festival  Solar_Irradiance  Year  Month  Month_sin  Month_cos  \
12         1             200.0  2026      1        0.5   0.866025   

    Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  Demand_Rolling_3  \
12       10150.0       10076.0       10087.0      10104.333333   

    Demand_Rolling_6  Demand_Rolling_12  
12           10823.0           10964.75  


In [93]:
jan_2026.head()

,Date,Electricity_Requirement,Temperature,Rainfall,Humidity,Festival,Solar_Irradiance,Year,Month,Month_sin,Month_cos,Demand_Lag_1,Demand_Lag_2,Demand_Lag_3,Demand_Rolling_3,Demand_Rolling_6,Demand_Rolling_12
12,2026-01-01,10067,25.5,38.5,73.5,1,200.0,2026,1,0.5,0.866025,10150.0,10076.0,10087.0,10104.333333,10823.0,10964.75


In [94]:
# ============================================
# JANUARY 2026 INPUT VARIABLES
# ============================================

jan_row = jan_2026

# Basic features
date = jan_row["Date"]
temperature = float(jan_row["Temperature"])
rainfall = float(jan_row["Rainfall"])
humidity = float(jan_row["Humidity"])
festival = int(jan_row["Festival"])
solar_irradiance = float(jan_row["Solar_Irradiance"])

#Calendar features
year = int(jan_row["Year"])
month = int(jan_row["Month"])
month_sin = float(jan_row["Month_sin"])
month_cos = float(jan_row["Month_cos"])

# Demand history features
demand_lag_1 = float(jan_row["Demand_Lag_1"])
demand_lag_2 = float(jan_row["Demand_Lag_2"])
demand_lag_3 = float(jan_row["Demand_Lag_3"])

demand_rolling_3 = float(jan_row["Demand_Rolling_3"])
demand_rolling_6 = float(jan_row["Demand_Rolling_6"])
demand_rolling_12 = float(jan_row["Demand_Rolling_12"])

# Actual January demand
# IMPORTANT: use this ONLY for final evaluation
actual_jan_demand = float(
    jan_row["Electricity_Requirement"]
)

print("========================================")
print("JANUARY 2026 VARIABLES")
print("========================================")

print("Date              :", date)
print("Temperature       :", temperature)
print("Rainfall          :", rainfall)
print("Humidity          :", humidity)
print("Festival          :", festival)
print("Solar Irradiance  :", solar_irradiance)

print("Year              :", year)
print("Month             :", month)
print("Month Sin         :", month_sin)
print("Month Cos         :", month_cos)

print("Demand Lag 1      :", demand_lag_1)
print("Demand Lag 2      :", demand_lag_2)
print("Demand Lag 3      :", demand_lag_3)
print("Rolling 3         :", demand_rolling_3)
print("Rolling 6         :", demand_rolling_6)
print("Rolling 12        :", demand_rolling_12)

print("Actual Jan Demand :", actual_jan_demand)
print("========================================")

JANUARY 2026 VARIABLES
Date              : 12   2026-01-01
Name: Date, dtype: datetime64[ns]
Temperature       : 25.5
Rainfall          : 38.5
Humidity          : 73.5
Festival          : 1
Solar Irradiance  : 200.0
Year              : 2026
Month             : 1
Month Sin         : 0.49999999999999994
Month Cos         : 0.8660254037844387
Demand Lag 1      : 10150.0
Demand Lag 2      : 10076.0
Demand Lag 3      : 10087.0
Rolling 3         : 10104.333333333334
Rolling 6         : 10823.0
Rolling 12        : 10964.75
Actual Jan Demand : 10067.0


C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_4884\23034611.py:9: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  temperature = float(jan_row["Temperature"])
C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_4884\23034611.py:10: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  rainfall = float(jan_row["Rainfall"])
C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_4884\23034611.py:11: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  humidity = float(jan_row["Humidity"])
C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_4884\23034611.py:12: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  festival = int(jan_row["F

In [95]:
# ========================================
# CREATE 3-MONTH SEQUENCE
# NOV 2025 → DEC 2025 → JAN 2026
# ========================================

sequence_dates = [
    "2025-11-01",
    "2025-12-01",
    "2026-01-01"
]

sequence_data = df[
    df["Date"].isin(pd.to_datetime(sequence_dates))
].sort_values("Date")

print(sequence_data[["Date"] + FEATURES])
print("\nShape:", sequence_data[FEATURES].shape)

         Date  Humidity    Rainfall  Solar_Irradiance  Temperature  Year  \
10 2025-11-01     83.51  206.957714        197.768678        25.32  2025   
11 2025-12-01     81.73   40.279714        207.785030        23.56  2025   
12 2026-01-01     73.50   38.500000        200.000000        25.50  2026   

       Month_sin  Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  \
10 -5.000000e-01   0.866025         0       10087.0       10972.0   
11 -2.449294e-16   1.000000         0       10076.0       10087.0   
12  5.000000e-01   0.866025         1       10150.0       10076.0   

    Demand_Lag_3  Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
10       11271.0      10776.666667      11259.666667                NaN  
11       10972.0      10378.333333      11050.333333                NaN  
12       10087.0      10104.333333      10823.000000           10964.75  

Shape: (3, 14)


In [96]:
print(df[df["Date"].isin(pd.to_datetime([
    "2025-11-01",
    "2025-12-01",
    "2026-01-01"
]))][
    ["Date", "Demand_Rolling_3",
     "Demand_Rolling_6", "Demand_Rolling_12"]
])

         Date  Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12
10 2025-11-01      10776.666667      11259.666667                NaN
11 2025-12-01      10378.333333      11050.333333                NaN
12 2026-01-01      10104.333333      10823.000000           10964.75


In [97]:
X_jan_2026.shape

(1, 14)

In [98]:
jan_sequence = df[
    (df["Date"] >= "2025-11-01") &
    (df["Date"] <= "2026-01-01")
].copy()

print(jan_sequence[FEATURES])
print("Shape:", jan_sequence[FEATURES].shape)

    Humidity    Rainfall  Solar_Irradiance  Temperature  Year     Month_sin  \
10     83.51  206.957714        197.768678        25.32  2025 -5.000000e-01   
11     81.73   40.279714        207.785030        23.56  2025 -2.449294e-16   
12     73.50   38.500000        200.000000        25.50  2026  5.000000e-01   

    Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  \
10   0.866025         0       10087.0       10972.0       11271.0   
11   1.000000         0       10076.0       10087.0       10972.0   
12   0.866025         1       10150.0       10076.0       10087.0   

    Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
10      10776.666667      11259.666667                NaN  
11      10378.333333      11050.333333                NaN  
12      10104.333333      10823.000000           10964.75  
Shape: (3, 14)


In [99]:
print(jan_sequence[FEATURES].isna().sum())

Humidity             0
Rainfall             0
Solar_Irradiance     0
Temperature          0
Year                 0
Month_sin            0
Month_cos            0
Festival             0
Demand_Lag_1         0
Demand_Lag_2         0
Demand_Lag_3         0
Demand_Rolling_3     0
Demand_Rolling_6     0
Demand_Rolling_12    2
dtype: int64


In [100]:
# Insert Demand_Rolling_12 values

df.loc[df["Date"] == "2025-11-01", "Demand_Rolling_12"] = 10853.67
df.loc[df["Date"] == "2025-12-01", "Demand_Rolling_12"] = 10903.83
df.loc[df["Date"] == "2026-01-01", "Demand_Rolling_12"] = 10964.75

In [102]:
# Create 3-month sequence for January 2026 prediction

jan_sequence = df[
    (df["Date"] >= "2025-11-01") &
    (df["Date"] <= "2026-01-01")
].copy()

# Select only the required features
X_jan = jan_sequence[FEATURES]

print(X_jan)
print("Shape:", X_jan.shape)

    Humidity    Rainfall  Solar_Irradiance  Temperature  Year     Month_sin  \
10     83.51  206.957714        197.768678        25.32  2025 -5.000000e-01   
11     81.73   40.279714        207.785030        23.56  2025 -2.449294e-16   
12     73.50   38.500000        200.000000        25.50  2026  5.000000e-01   

    Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  \
10   0.866025         0       10087.0       10972.0       11271.0   
11   1.000000         0       10076.0       10087.0       10972.0   
12   0.866025         1       10150.0       10076.0       10087.0   

    Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
10      10776.666667      11259.666667           10853.67  
11      10378.333333      11050.333333           10903.83  
12      10104.333333      10823.000000           10964.75  
Shape: (3, 14)


In [103]:
print(X_jan.isna().sum())

Humidity             0
Rainfall             0
Solar_Irradiance     0
Temperature          0
Year                 0
Month_sin            0
Month_cos            0
Festival             0
Demand_Lag_1         0
Demand_Lag_2         0
Demand_Lag_3         0
Demand_Rolling_3     0
Demand_Rolling_6     0
Demand_Rolling_12    0
dtype: int64


In [104]:
X_jan = X_jan.values.reshape(1, 3, 14)

print("LSTM Input Shape:", X_jan.shape)

LSTM Input Shape: (1, 3, 14)


In [106]:
jan_prediction_scaled = model.predict(X_jan, verbose=0)

jan_prediction = target_scaler.inverse_transform(
    jan_prediction_scaled
)[0][0]

print("========================================")
print("JANUARY 2026 DEMAND PREDICTION")
print("========================================")
print(f"Predicted Demand : {jan_prediction:.2f}")
print(f"Actual Demand    : 10067.00")
print("========================================")

NameError: name 'target_scaler' is not defined

In [107]:
print(type(model))

<class 'keras.src.models.sequential.Sequential'>


In [110]:
from sklearn.preprocessing import MinMaxScaler

In [112]:
import joblib
import numpy as np
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("Demand_LSTM.keras")

# Load scalers used during training
feature_scaler = joblib.load("Demand_LSTM_X_Scaler.pkl")
target_scaler = joblib.load("Demand_LSTM_y_Scaler.pkl")

print("Model and scalers loaded successfully")

Model and scalers loaded successfully


In [114]:
# January sequence
X_jan = jan_sequence[FEATURES].values

# Scale using the SAME feature scaler from training
X_jan_scaled = feature_scaler.transform(X_jan)

# Reshape: samples, timesteps, features
X_jan_scaled = X_jan_scaled.reshape(1, 3, 14)

# Predict
jan_prediction_scaled = model.predict(X_jan_scaled, verbose=0)

# Convert prediction back to actual MU
jan_prediction = target_scaler.inverse_transform(
    jan_prediction_scaled
)[0][0]

print("========================================")
print("JANUARY 2026 ELECTRICITY DEMAND")
print("========================================")
print(f"Predicted Demand : {jan_prediction:.2f} MU")
print(f"Actual Demand    : 10067.00 MU")
print("========================================")

JANUARY 2026 ELECTRICITY DEMAND
Predicted Demand : 11047.51 MU
Actual Demand    : 10067.00 MU


In [115]:
actual = 10067.00
predicted = 11047.51

error = abs(actual - predicted)
ape = (error / actual) * 100

print(f"Absolute Error : {error:.2f} MU")
print(f"APE            : {ape:.2f}%")

Absolute Error : 980.51 MU
APE            : 9.74%


In [116]:
feb_2026 = df[df["Date"] == "2026-02-01"].copy()

print(feb_2026)

         Date  Electricity_Requirement  Temperature  Rainfall  Humidity  \
13 2026-02-01                    10125         26.5      12.2      69.5   

    Festival  Solar_Irradiance  Year  Month  Month_sin  Month_cos  \
13         0             225.0  2026      2   0.866025        0.5   

    Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  Demand_Rolling_3  \
13       10067.0       10150.0       10076.0      10097.666667   

    Demand_Rolling_6  Demand_Rolling_12  
13      10437.166667       10987.416667  


In [117]:
print(feb_2026[FEATURES])

    Humidity  Rainfall  Solar_Irradiance  Temperature  Year  Month_sin  \
13      69.5      12.2             225.0         26.5  2026   0.866025   

    Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  \
13        0.5         0       10067.0       10150.0       10076.0   

    Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
13      10097.666667      10437.166667       10987.416667  


In [118]:
print(feb_2026[FEATURES].isna().sum())

Humidity             0
Rainfall             0
Solar_Irradiance     0
Temperature          0
Year                 0
Month_sin            0
Month_cos            0
Festival             0
Demand_Lag_1         0
Demand_Lag_2         0
Demand_Lag_3         0
Demand_Rolling_3     0
Demand_Rolling_6     0
Demand_Rolling_12    0
dtype: int64


In [119]:
feb_sequence = df[
    df["Date"].isin([
        "2025-12-01",
        "2026-01-01",
        "2026-02-01"
    ])
].sort_values("Date")

print(feb_sequence[FEATURES])
print("Shape:", feb_sequence[FEATURES].shape)

    Humidity   Rainfall  Solar_Irradiance  Temperature  Year     Month_sin  \
11     81.73  40.279714         207.78503        23.56  2025 -2.449294e-16   
12     73.50  38.500000         200.00000        25.50  2026  5.000000e-01   
13     69.50  12.200000         225.00000        26.50  2026  8.660254e-01   

    Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  \
11   1.000000         0       10076.0       10087.0       10972.0   
12   0.866025         1       10150.0       10076.0       10087.0   
13   0.500000         0       10067.0       10150.0       10076.0   

    Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
11      10378.333333      11050.333333       10903.830000  
12      10104.333333      10823.000000       10964.750000  
13      10097.666667      10437.166667       10987.416667  
Shape: (3, 14)


In [120]:
X_feb = feb_sequence[FEATURES].values

X_feb = X_feb.reshape(1, 3, 14)

print("LSTM Input Shape:", X_feb.shape)

LSTM Input Shape: (1, 3, 14)


In [130]:
# Historical 3-month sequence for predicting February 2026
feb_sequence = df[
    df["Date"].isin([
        "2025-11-01",
        "2025-12-01",
        "2026-01-01"
    ])
].sort_values("Date")

print(feb_sequence[FEATURES])
print("Shape:", feb_sequence[FEATURES].shape)

    Humidity    Rainfall  Solar_Irradiance  Temperature  Year     Month_sin  \
10     83.51  206.957714        197.768678        25.32  2025 -5.000000e-01   
11     81.73   40.279714        207.785030        23.56  2025 -2.449294e-16   
12     73.50   38.500000        200.000000        25.50  2026  5.000000e-01   

    Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  \
10   0.866025         0       10087.0       10972.0       11271.0   
11   1.000000         0       10076.0       10087.0       10972.0   
12   0.866025         1       10150.0       10076.0       10087.0   

    Demand_Rolling_3  Demand_Rolling_6  Demand_Rolling_12  
10      10776.666667      11259.666667           10853.67  
11      10378.333333      11050.333333           10903.83  
12      10104.333333      10823.000000           10964.75  
Shape: (3, 14)


In [131]:
# Prepare LSTM input
X_feb = feb_sequence[FEATURES].values.reshape(1, 3, 14)

print("X_feb shape:", X_feb.shape)

X_feb shape: (1, 3, 14)


In [136]:
# ==========================================
# FEBRUARY 2026 DEMAND PREDICTION
# ==========================================

# Your current X_feb contains RAW values
# Scale them using the training feature scaler

X_feb_raw = X_feb.reshape(3, 14)

X_feb_scaled = feature_scaler.transform(X_feb_raw)

# Reshape for LSTM
X_feb = X_feb_scaled.reshape(1, 3, 14)

print("X_feb scaled shape:", X_feb.shape)

# Scaled prediction
feb_prediction_scaled = model.predict(X_feb, verbose=0)

print("Scaled prediction:", feb_prediction_scaled)

# Convert prediction back to MU
feb_prediction = target_scaler.inverse_transform(
    feb_prediction_scaled
)[0][0]

print("========================================")
print("FEBRUARY 2026 ELECTRICITY DEMAND")
print("========================================")
print(f"Predicted Demand : {feb_prediction:.2f} MU")
print("========================================")

X_feb scaled shape: (1, 3, 14)
Scaled prediction: [[0.9129672]]
FEBRUARY 2026 ELECTRICITY DEMAND
Predicted Demand : 11047.51 MU


In [137]:
print("JANUARY INPUT:")
print(X_jan)

print("\nFEBRUARY INPUT:")
print(X_feb)

JANUARY INPUT:
[[ 8.35100000e+01  2.06957714e+02  1.97768678e+02  2.53200000e+01
   2.02500000e+03 -5.00000000e-01  8.66025404e-01  0.00000000e+00
   1.00870000e+04  1.09720000e+04  1.12710000e+04  1.07766667e+04
   1.12596667e+04  1.08536700e+04]
 [ 8.17300000e+01  4.02797143e+01  2.07785030e+02  2.35600000e+01
   2.02500000e+03 -2.44929360e-16  1.00000000e+00  0.00000000e+00
   1.00760000e+04  1.00870000e+04  1.09720000e+04  1.03783333e+04
   1.10503333e+04  1.09038300e+04]
 [ 7.35000000e+01  3.85000000e+01  2.00000000e+02  2.55000000e+01
   2.02600000e+03  5.00000000e-01  8.66025404e-01  1.00000000e+00
   1.01500000e+04  1.00760000e+04  1.00870000e+04  1.01043333e+04
   1.08230000e+04  1.09647500e+04]]

FEBRUARY INPUT:
[[[ 0.82096346  0.51688766  0.96932995  0.22281879  1.28572722
    0.25        0.9330127   0.          0.67977669  0.89463466
    0.9672251   0.89432359  1.06461239  1.30585474]
  [ 0.7670404   0.09901307  1.05166064 -0.01342279  1.28572722
    0.5         1.         

In [138]:
# January raw input
X_jan_raw = jan_sequence[FEATURES].values

# Scale January using the SAME feature scaler
X_jan_scaled = feature_scaler.transform(X_jan_raw)

# LSTM shape
X_jan = X_jan_scaled.reshape(1, 3, 14)

# Predict January
jan_prediction_scaled = model.predict(X_jan, verbose=0)

jan_prediction = target_scaler.inverse_transform(
    jan_prediction_scaled
)[0][0]

print("January scaled prediction:", jan_prediction_scaled)
print(f"January prediction: {jan_prediction:.2f} MU")

January scaled prediction: [[0.9129672]]
January prediction: 11047.51 MU


In [141]:
jan_scaled = model.predict(X_jan, verbose=0)
feb_scaled = model.predict(X_feb, verbose=0)

print("January scaled prediction:", jan_scaled)
print("February scaled prediction:", feb_scaled)

January scaled prediction: [[0.9129672]]
February scaled prediction: [[0.9129672]]


In [143]:
print("X_jan id:", id(X_jan))
print("X_feb id:", id(X_feb))

print("\nJAN first value:")
print(X_jan[0, 0, 0])

print("\nFEB first value:")
print(X_feb[0, 0, 0])

X_jan id: 1879603055184
X_feb id: 1879592452816

JAN first value:
0.8209634551405909

FEB first value:
0.8209634551405909
